

## Setup

In [ ]:
%%capture

%pip install "git+https://github.com/somombo/impalab.git@python-sdk#subdirectory=python-sdk"

In [ ]:
from impalab_py.colab import colab_repo_sync

colab_repo_sync(
  repo_url="https://github.com/somombo/sort-bench.git",
  repo_subdir="lab",
)


In [ ]:
from impalab_py import Impa
import json

impa = Impa(
  root_dir="..",
)


In [ ]:
from impalab_py.colab import is_colab_env
import os

if is_colab_env() or (os.getenv("GITHUB_ACTIONS") == "true"): 
  impa.download_executable()

  from env_setup import *

  setup_lean()

Let us run a warmup round of the benchmark to pre-build build all the executables

In [ ]:
_build_result = impa.build(
  components_dir="./components",
  include=[
    'somombo_unifshuffle_multi',
    'lean',
  ]
)
assert _build_result, "Initial build failed"

In [ ]:
_test_results = impa.run(
  generator={
    "name": "somombo_unifshuffle_multi",
    "seed": 42,
    "args": [json.dumps([
      {
        "cardinality": 19,
        "multiplicity": 13,
        "swaps": 2,
        "descending": True,
        "runs": 7,
      }, 
      {
        "cardinality": 17,
        "multiplicity": 11,
        "swaps": None,
        "descending": False,
        "runs": 5,
      }
    ])],
  }, 
  reps=3, 
  tasks=[
    {'executor': 'lean', 'args': ['Somombo.qs.bentleyMcIlroy.classic']},
    {'executor': 'lean', 'args': ['List.mergeSort']},
  ],
  attributes={
    "experiment_name": "Warmup-Experiment",
  },
)

## Experiment Config

In [ ]:
from exploration import ExplorerFromLabDf

ExplorerFromLabDf(_test_results).get_clean_data().head()

In [ ]:
from sample_factors import sample_spaced, get_factors, get_hcns

In [ ]:
import secrets

tasks = [
  {'executor': 'lean', 'args': ['Somombo.qs.hoare.classic']},
  
  {'executor': 'lean', 'args': ['Array.insertionSort']},
  {'executor': 'lean', 'args': ['Somombo.Array.insertionSort_without_fuel']},
  {'executor': 'lean', 'args': ['Somombo.Vector.insertionSort']},
]
# 
generator = {
  "name": "somombo_unifshuffle_multi",
  # "seed":  9352554831993720236, 
  "seed":  secrets.randbits(64), 
}

attributes = {
  "study": "insertionSort_study"
}


generator

## Experiment 1: Cardinality (Pre-sorted in Ascending Order)

In [ ]:
# _cards = sample_spaced(
#     10,
#     start=1000,
#     stop=10_000,
#     geometric=False,
# )

runs = 10
reps = 5

experiment_name = "Experiment 1: Varying Cardinality"

_cards = sample_spaced(
    5,
    # start=1,
    # stop=1000,

    start=10,
    stop=30_000,
    geometric=True,
)
# _cardinalities = [1000, 10_000, 100_000, 1_000_000, 10_000_000]
print("Geometric Progression of Cardinalities to test:", _cards)

params_list = [
    {"cardinality": card, "multiplicity": 1, "descending": False, 'swaps': None, "runs": runs} for card in _cards
]


total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')

In [ ]:
results1 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp1 = ExplorerFromLabDf(results1)

In [ ]:
exp1.plot_distributions( 
  'gen.cardinality', 
  title=f'Performance Distributions vs. Cardinality (Unique Elements: Multiplicity=1)',
  normalized=False,
  ylog=True,
)

In [ ]:
exp1.plot_trends_interactive( 
    'gen.cardinality', 
    f'Median Sort Performance vs. Cardinality (Unique Elements: Multiplicity=1)',
    normalized=True,
    xlog=True,
    ylog=True,
)

## Experiment 2: Multiplicity (Fixed Size Pre-sorted in Ascending Order)

In [ ]:
runs = 5
reps = 5

# fixed_size = 10080
fixed_size = 15120
# fixed_size = 20160
# fixed_size = 27720

experiment_name = f"Experiment 2 (Varying Multiplicity, Fixed Size = {fixed_size})"

assert fixed_size in get_hcns(), f"The chosen Fixed Size {fixed_size} is not a Highly Composite Number (HCN)"
print(f"Fixed Size: {fixed_size}", )


_multis = sample_spaced(
    count=7,
    population=get_factors(fixed_size),
    geometric=True,
    # strategy="furthest",
)

print(f'Approximately Geometric Progression of Multiplicities to test that are factors of Fixed Size = {fixed_size} with len={len(_multis)} sample points: \n    {_multis}')

# Vary multiplicity and calculate corresponding cardinality
params_list = []
for multi in _multis:
    if fixed_size % multi == 0: # Ensure multiplicity is an integer
        card = fixed_size // multi
        params_list.append({
            'cardinality': card,
            'multiplicity': multi,
            'descending': False,
            'swaps': None,
            'runs': runs,
        })

# assert  len(params_list) == 17

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')

In [ ]:
results2 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp2 = ExplorerFromLabDf(results2)

In [ ]:
exp2.plot_distributions(
    'gen.multiplicity',
    title=f'Performance Distribution vs. Multiplicity (Fixed Size = {fixed_size})',
    normalized=False,
    ylog=True,
)

In [ ]:
exp2.plot_trends_interactive(
    'gen.multiplicity',
    f'Median Sort Performance vs. Multiplicity (Fixed Size = {fixed_size})',
    normalized=False,
    xlog=True,
    ylog=False,
)

## Experiment 3: Adaptability (Pre-sorted in Ascending Order)

In [ ]:
runs = 10
reps = 5

fixed_size = 20000
experiment_name = "Experiment 3 (Varying Swaps)"

_swaps = [
  0, 
  *sample_spaced(
    count=14,
    start=1,
    stop=fixed_size,
    geometric=True,
    # strategy="furthest",
  )
]
print(f"swaps ({len(_swaps)}):", _swaps)


params_list = [
  {'swaps': swap, 'cardinality': fixed_size, 'multiplicity': 1, 'descending': False, "runs": runs} for swap in _swaps
]

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')


In [ ]:
results3 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp3 = ExplorerFromLabDf(results3)

In [ ]:
exp3.plot_distributions(
  'gen.swaps', 
  title=f'Performance Distribution vs. Pre-sortedness (Size = {fixed_size}, All Unique Values)',
  normalized=False,
  ylog=True,
)

In [ ]:
exp3.plot_trends_interactive(
  'gen.swaps',
  f'Median Sort Performance vs. Pre-sortedness (Size = {fixed_size}, All Unique Values)', 
  normalized=False, 
  xlog=True, 
  ylog=True,
)

## Experiment 4: Cardinality (Pre-sorted in Descending Order)

In [ ]:
runs = 5
reps = 5

experiment_name = "Experiment 4 (Varying Cardinality Descending)"

# _cards = sample_spaced(
#     10,
#     start=1000,
#     stop=10_000,
#     geometric=False,
# )


_cards = sample_spaced(
    5,
    start=10,
    stop=30_000,
    geometric=True,
)
print("Geometric Progression of Cardinalities to test:", _cards)

params_list = [
    {"cardinality": card, "multiplicity": 1, "swaps": 2, "descending": True, "runs": runs} for card in _cards
]

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')

In [ ]:
results4 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp4 = ExplorerFromLabDf(results4)

In [ ]:
exp4.plot_distributions(
  'gen.cardinality', 
  title=f'Performance Distributions vs. Cardinality (Unique Elements: Multiplicity=1)',
  normalized=False,
  ylog=True,
)

In [ ]:
exp4.plot_trends_interactive(
    'gen.cardinality', 
    f'Median Sort Performance vs. Cardinality (Unique Elements: Multiplicity=1)',
    normalized=True,
    xlog=True,
    ylog=True,
)

## Experiment 5: Multiplicity (Fixed Size Pre-sorted in Descending Order)

In [ ]:
runs = 10
reps = 5

fixed_size = 10080
# fixed_size = 15120
# fixed_size = 20160
# fixed_size = 25200
# fixed_size = 27720

experiment_name = f"Experiment 5 (Varying Multiplicity Descending, Fixed Size = {fixed_size})"

assert fixed_size in get_hcns(), f"The chosen Fixed Size {fixed_size} is not a Highly Composite Number (HCN)"
print(f"Fixed Size: {fixed_size}")


_multis = sample_spaced(
    count=9,
    population=get_factors(fixed_size),
    geometric=True,
    strategy="furthest",
)

print(f'Approximately Geometric Progression of Multiplicities to test that are factors of Fixed Size = {fixed_size} with len={len(_multis)} sample points: \n    {_multis}')

# Vary multiplicity and calculate corresponding cardinality
params_list = []
for multi in _multis:
    if fixed_size % multi == 0: # Ensure multiplicity is an integer
        card = fixed_size // multi
        params_list.append({
            'cardinality': card,
            'multiplicity': multi,
            'descending': True,
            "swaps": 2,
            'runs': runs,
        })

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')

In [ ]:
results5 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp5 = ExplorerFromLabDf(results5)

In [ ]:
exp5.plot_distributions(
    'gen.multiplicity',
    title=f'Performance Distribution vs. Multiplicity (Fixed Size = {fixed_size})',
    normalized=False,
    ylog=False,
)

In [ ]:
exp5.plot_trends_interactive(
    'gen.multiplicity',
    f'Median Sort Performance vs. Multiplicity (Fixed Size = {fixed_size})',
    normalized=False,
    xlog=True,
    ylog=False,
)

## Experiment 6: Adaptability (Pre-sorted in Descending Order)

In [ ]:
runs = 5
reps = 5

fixed_size = 10_000
experiment_name = "Experiment 6 (Varying Swaps Descending)"

_swaps = [
  0, 
  *sample_spaced(
    count=9,
    start=1,
    stop=fixed_size,
    geometric=True,
  )
]
print(f"swaps ({len(_swaps)}):", _swaps)


params_list = [
  {'swaps': swaps, 'cardinality': fixed_size, 'multiplicity': 1, "descending": True, "runs": runs} for swaps in _swaps
]

total_data_points = len(tasks) * len(params_list) * reps * runs

import pandas as pd
pd.DataFrame(params_list).style.hide(axis='index')

In [ ]:
results6 = impa.run(
  pbar_total=total_data_points,
  tasks=tasks,
  generator = {
    **generator,
    "args": [json.dumps(params_list)],
  },
  reps = reps,
  attributes = {
    **attributes,
    "experiment_name": experiment_name,
  }
)

exp6 = ExplorerFromLabDf(results6)

In [ ]:
exp6.plot_distributions(
  'gen.swaps', 
  title=f'Performance Distribution vs. Pre-sortedness (Size = {fixed_size}, All Unique Values)',
  normalized=False,
  ylog=False,
)

In [ ]:
exp6.plot_trends_interactive(
  'gen.swaps',
  f'Median Sort Performance vs. Pre-sortedness (Size = {fixed_size}, All Unique Values)', 
  normalized=False, 
  xlog=True, 
  ylog=True,
)

## Export Results of Study

In [ ]:
import os
from impalab_py import jsonl

os.makedirs("../data", exist_ok=True)

with open(f'../data/{attributes["study"]}.jsonl', 'w', encoding='utf-8') as f:

  f.write(jsonl.join([results1, results2, results3, results4, results5, results6]))

In [ ]:
with open(f'../data/{attributes["study"]}.jsonl', 'r', encoding='utf-8') as f:

  print(f.read())
